In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import tensorflow as tf
import pandas as pd 


In [ ]:
df=pd.read_csv('churn_dataset.csv')
df.head

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['Gender']=le.fit_transform(df['Gender'])


In [8]:
df = pd.get_dummies(df, columns=['Geography'], dtype=int)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


In [11]:
from sklearn.preprocessing import OneHotEncoder
oh=OneHotEncoder()
df2=pd.read_csv('churn_dataset.csv')
oh.fit_transform(df2[['Geography']])


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [12]:
x=df.drop('Exited',axis=1)
y=df['Exited']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
for col in x_train.columns:
    x_train[col]=(x_train[col]-x_train[col].min())/(x_train[col].max()-x_train[col].min())
    x_test[col]=(x_test[col]-x_test[col].min())/(x_test[col].max()-x_test[col].min())
x_train.head()



,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
9069,0.538,0.0,0.189189,0.4,0.699113,0.333333,1.0,1.0,0.864027,0.0,0.0,1.0
2603,0.586,0.0,0.216216,0.7,0.639407,0.000000,0.0,1.0,0.942778,0.0,1.0,0.0
7738,0.422,1.0,0.202703,0.6,0.000000,0.333333,0.0,0.0,0.868470,1.0,0.0,0.0
1579,0.536,1.0,0.310811,0.8,0.150271,0.000000,1.0,1.0,0.979011,0.0,1.0,0.0
5058,0.728,1.0,0.256757,0.9,0.591742,0.333333,0.0,1.0,0.756406,1.0,0.0,0.0


In [14]:
y_train

9069    1
2603    0
7738    0
1579    0
5058    0
       ..
5734    0
5191    0
5390    1
860     1
7270    0
Name: Exited, Length: 7000, dtype: int64

In [16]:
x_train.isnull().sum()

CreditScore          0
Gender               0
Age                  0
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Geography_France     0
Geography_Germany    0
Geography_Spain      0
dtype: int64

In [18]:
y_train.value_counts()

Exited
0    5547
1    1453
Name: count, dtype: int64

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [24]:
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping
model=Sequential(
    [
        Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
        Dense(32,activation='relu'),
        Dense(1,activation='sigmoid')
    ]
)


/Users/shivang/Desktop/tensorflow/myenv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
import datetime
opt=tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])
EarlyStopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
TensorBoard_call=TensorBoard(log_dir=log_dir,histogram_freq=0)
history=model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=100,callbacks=[EarlyStopping,TensorBoard_call])



Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 945us/step - accuracy: 0.7929 - loss: 0.4927 - val_accuracy: 0.8093 - val_loss: 0.4473
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - accuracy: 0.8011 - loss: 0.4519 - val_accuracy: 0.8137 - val_loss: 0.4272
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 631us/step - accuracy: 0.8126 - loss: 0.4345 - val_accuracy: 0.8177 - val_loss: 0.4136
Epoch 4/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 634us/step - accuracy: 0.8197 - loss: 0.4171 - val_accuracy: 0.8330 - val_loss: 0.3904
Epoch 5/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 635us/step - accuracy: 0.8324 - loss: 0.3936 - val_accuracy: 0.8420 - val_loss: 0.3706
Epoch 6/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 608us/step - accuracy: 0.8397 - loss: 0.3761 - val_accuracy: 0.8497 - val_loss: 0.3612
Epoch 7/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - accuracy: 0.8433 - loss: 0.3669 - val_accuracy: 0.8530 - val_loss: 0.3552
Epoch 8/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step - accuracy: 0.8477 - loss: 0

In [26]:
import pickle
with open('label.pkl','wb') as f :
    pickle.dump(le,f)

with open('ohe.pkl','wb') as f:
    pickle.dump(oh,f)
    
       
    
    
    
    

In [27]:
model.save('model.h5')


In [31]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [40]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6007 (pid 10761), started 0:03:14 ago. (Use '!kill 10761' to kill it.)

In [41]:
from tensorflow.keras.models import load_model
model5=load_model('model.h5') 
with open('label.pkl','rb') as f:
    le=pickle.load(f)
with open('ohe.pkl','rb') as f:
    oh=pickle.load(f)

In [63]:
df2.columns
df2

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [84]:
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}



In [64]:
df2.drop(['RowNumber','CustomerId','Surname'], axis=1, inplace=True)
df2.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [67]:
ls=LabelEncoder()
df2['Gender']=ls.fit_transform(df2['Gender'])
df2.head()
df2=pd.get_dummies(df2,columns=['Geography'], dtype=int)
df2.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


In [69]:
from sklearn.model_selection import train_test_split
x=df2.drop('Exited',axis=1)
y=df2['Exited']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)



In [71]:
x_train


array([[-0.34459497, -1.09823226, -0.65674999, ..., -1.00171576,
        -0.57559072,  1.73073215],
       [-0.09518109, -1.09823226, -0.46637979, ..., -1.00171576,
         1.73734559, -0.57779016],
       [-0.94734518,  0.91055421, -0.56156489, ...,  0.99828718,
        -0.57559072, -0.57779016],
       ...,
       [ 0.86090545, -1.09823226, -0.08563939, ...,  0.99828718,
        -0.57559072, -0.57779016],
       [ 0.15423279,  0.91055421,  0.39028611, ...,  0.99828718,
        -0.57559072, -0.57779016],
       [ 0.46600014,  0.91055421,  1.1517669 , ..., -1.00171576,
         1.73734559, -0.57779016]], shape=(7000, 12))

In [73]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model2=Sequential(
    [
        Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
        Dense(32,activation='relu'),
        Dense(1,activation='sigmoid')
    ]
) 

/Users/shivang/Desktop/tensorflow/myenv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [75]:
import datetime
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping
opt=tf.keras.optimizers.Adam(learning_rate=0.001)
model2.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])
earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
TensorBoard_call=TensorBoard(log_dir=log_dir,histogram_freq=1)
history=model2.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=100,callbacks=[earlystopping,TensorBoard_call])

Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7841 - loss: 0.4763 - val_accuracy: 0.8267 - val_loss: 0.4009
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 692us/step - accuracy: 0.8394 - loss: 0.3919 - val_accuracy: 0.8573 - val_loss: 0.3615
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 677us/step - accuracy: 0.8496 - loss: 0.3602 - val_accuracy: 0.8613 - val_loss: 0.3460
Epoch 4/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 650us/step - accuracy: 0.8566 - loss: 0.3497 - val_accuracy: 0.8620 - val_loss: 0.3435
Epoch 5/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 633us/step - accuracy: 0.8534 - loss: 0.3449 - val_accuracy: 0.8647 - val_loss: 0.3398
Epoch 6/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 643us/step - accuracy: 0.8583 - loss: 0.3414 - val_accuracy: 0.8647 - val_loss: 0.3369
Epoch 7/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 721us/step - accuracy: 0.8599 - loss: 0.3374 - val_accuracy: 0.8640 - val_loss: 0.3372
Epoch 8/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step - accuracy: 0.8580 - loss: 0.3

In [76]:
model2.save('model2.h5')


In [77]:
with open('sc.pkl','wb') as f:
    pickle.dump(sc,f)

In [89]:
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}



In [ ]:
import pandas as pd
import numpy as np

# 1. Start with your fresh input
input_df = pd.DataFrame([input_data])

# 2. Encode Gender (using the LabelEncoder 'le' you already have)
input_df['Gender'] = le.transform(input_df['Gender'])

# 3. Use the OHE on the Geography column
# Note: ohe should be the object you fitted during training
geo_encoded = oh.transform(input_df[['Geography']]).toarray()

geo_cols = oh.get_feature_names_out(['Geography'])

# 5. Create a DataFrame from the encoded features
geo_df = pd.DataFrame(geo_encoded, columns=geo_cols)

# 6. Combine and drop the old text column
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_df], axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [95]:
input_df_scaled = sc.transform(input_df)
input_df_scaled

array([[-0.54204762,  0.91055421,  0.10473081, -0.68894811, -0.26230046,
         0.81966266,  0.64598061,  0.97071435, -0.88153859,  0.99828718,
        -0.57559072, -0.57779016]])

In [97]:
prediction = model2.predict(input_df_scaled)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
[[0.02536682]]


In [98]:
if prediction[0][0]>0.5:
    print("the customer is likely to churn")
else:
    print("the customer is not likely to churn")    

the customer is not likely to churn


In [99]:
%load_ext tensorboard


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [100]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6007 (pid 10761), started 2:14:15 ago. (Use '!kill 10761' to kill it.)